# EA2 - Actividad 2.2: Transformacion y Limpieza de Datos

## Objetivos
- Detectar y manejar valores nulos (nulls) en DataFrames
- Identificar y eliminar registros duplicados
- Realizar casting y conversion de tipos de datos
- Crear funciones definidas por el usuario (UDFs)
- Combinar multiples fuentes de datos con join y union

## Conceptos Clave

### Calidad de Datos en un Pipeline ETL

En un pipeline ETL (Extract, Transform, Load), la fase de **transformacion** es donde se limpia y prepara los datos. Los problemas mas comunes son:

| Problema | Descripcion | Solucion |
|----------|-------------|----------|
| **Valores nulos** | Campos vacios o sin datos | `dropna()`, `fillna()` |
| **Duplicados** | Registros repetidos | `dropDuplicates()` |
| **Tipos incorrectos** | Columna numerica leida como string | `cast()`, `to_date()` |
| **Formatos inconsistentes** | Mezcla de mayusculas/minusculas | `lower()`, `upper()`, `trim()` |
| **Valores fuera de rango** | Datos que no tienen sentido | `filter()`, `when()` |

### UDFs (User Defined Functions)

Cuando la logica de transformacion es compleja y no se puede expresar con funciones built-in de Spark, se pueden crear **UDFs**. Sin embargo, las UDFs son mas lentas que las funciones nativas porque Spark no puede optimizarlas.

**Regla:** Siempre preferir funciones nativas de Spark (`F.when()`, `F.col()`, etc.) sobre UDFs cuando sea posible.

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("EA2_02_transformacion_limpieza") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

## 1. Cargar los Datos

Trabajaremos con dos datasets: `sales.csv` (ventas semanales por tienda y departamento) y `stores.csv` (informacion de las tiendas).

In [ ]:
# Leer los datasets
df_sales = spark.read.csv("/home/jovyan/datos/sales.csv", header=True, inferSchema=True)
df_stores = spark.read.csv("/home/jovyan/datos/stores.csv", header=True, inferSchema=True)

print("=== Sales ===")
df_sales.printSchema()
df_sales.show(5)
print(f"Filas en sales: {df_sales.count()}")

print("\n=== Stores ===")
df_stores.printSchema()
df_stores.show(5)
print(f"Filas en stores: {df_stores.count()}")

## 2. Detectar Valores Nulos

Antes de limpiar, necesitamos saber **donde** estan los problemas. Esta consulta cuenta los nulls por columna.

In [ ]:
# Contar valores nulos por columna en sales
print("Valores nulos por columna en sales:")
df_sales.select(
    [F.count(F.when(F.isnull(c), c)).alias(c) for c in df_sales.columns]
).show()

# Tambien podemos ver el porcentaje de nulls
total_filas = df_sales.count()
print("Porcentaje de nulls por columna:")
df_sales.select(
    [F.round(F.count(F.when(F.isnull(c), c)) / total_filas * 100, 2).alias(c) for c in df_sales.columns]
).show()

## 3. Manejar Valores Nulos

Hay dos estrategias principales:
- **Eliminar filas** con `dropna()`: cuando los nulls son pocos y no queremos inventar datos
- **Rellenar valores** con `fillna()`: cuando queremos mantener las filas y usar un valor por defecto

In [ ]:
# Estrategia 1: Eliminar filas con nulls en columnas criticas
df_sin_nulls = df_sales.dropna(subset=["Weekly_Sales"])
print(f"Filas originales: {df_sales.count()}")
print(f"Filas despues de dropna: {df_sin_nulls.count()}")
print(f"Filas eliminadas: {df_sales.count() - df_sin_nulls.count()}")

In [ ]:
# Estrategia 2: Rellenar nulls con valores por defecto
df_rellenado = df_sales.fillna({
    "Weekly_Sales": 0,
    "IsHoliday": False
})

# Verificar que ya no hay nulls en esas columnas
print("Nulls despues de fillna:")
df_rellenado.select(
    [F.count(F.when(F.isnull(c), c)).alias(c) for c in df_rellenado.columns]
).show()

## 4. Detectar y Eliminar Duplicados

Los duplicados pueden distorsionar completamente los resultados de un analisis.

In [ ]:
# Detectar duplicados
total = df_sales.count()
sin_duplicados = df_sales.dropDuplicates().count()

print(f"Total de filas: {total}")
print(f"Filas unicas: {sin_duplicados}")
print(f"Duplicados encontrados: {total - sin_duplicados}")

In [ ]:
# Tambien podemos buscar duplicados en columnas especificas
# Por ejemplo, duplicados por Store + Dept + Date
sin_dup_parcial = df_sales.dropDuplicates(["Store", "Dept", "Date"]).count()
print(f"Filas unicas por (Store, Dept, Date): {sin_dup_parcial}")
print(f"Duplicados parciales: {total - sin_dup_parcial}")

## 5. Conversion de Tipos (Casting)

A veces los tipos inferidos no son correctos. Podemos convertirlos manualmente.

In [ ]:
# Ejemplo: convertir columna Date de string a tipo fecha
df_con_fecha = df_sales.withColumn(
    "Date_parsed", 
    F.to_date("Date", "yyyy-MM-dd")
)

# Verificar el cambio
df_con_fecha.select("Date", "Date_parsed").show(5)
df_con_fecha.printSchema()

In [ ]:
# Otros castings comunes
df_cast = df_sales \
    .withColumn("Weekly_Sales_int", F.col("Weekly_Sales").cast(IntegerType())) \
    .withColumn("Store_str", F.col("Store").cast(StringType()))

df_cast.select("Weekly_Sales", "Weekly_Sales_int", "Store", "Store_str").show(5)
df_cast.printSchema()

## 6. Funciones Definidas por el Usuario (UDFs)

Las UDFs permiten aplicar logica personalizada de Python a cada fila del DataFrame.

**Importante:** Las UDFs son mas lentas que las funciones nativas de Spark porque:
1. Serializan datos de JVM a Python y viceversa
2. Spark no puede optimizar el codigo Python

Usar solo cuando no hay una funcion nativa equivalente.

In [ ]:
from pyspark.sql.functions import udf

# Definir una UDF que categoriza el monto de ventas
@udf(returnType=StringType())
def categorizar_venta(monto):
    if monto is None:
        return "Sin datos"
    elif monto > 50000:
        return "Alta"
    elif monto > 20000:
        return "Media"
    else:
        return "Baja"

# Aplicar la UDF
df_categorizado = df_sales.withColumn("categoria", categorizar_venta("Weekly_Sales"))
df_categorizado.select("Store", "Dept", "Weekly_Sales", "categoria").show(10)

In [ ]:
# Ver la distribucion de categorias
df_categorizado.groupBy("categoria").count().orderBy("count", ascending=False).show()

## 7. Alternativa Nativa: when/otherwise

Para logica condicional simple, **siempre preferir `F.when().otherwise()`** sobre UDFs. Es mas rapido porque Spark puede optimizarlo internamente.

In [ ]:
# Misma logica que la UDF pero con funciones nativas (mas rapido)
df_nativo = df_sales.withColumn(
    "categoria",
    F.when(F.col("Weekly_Sales").isNull(), "Sin datos")
     .when(F.col("Weekly_Sales") > 50000, "Alta")
     .when(F.col("Weekly_Sales") > 20000, "Media")
     .otherwise("Baja")
)

df_nativo.select("Store", "Dept", "Weekly_Sales", "categoria").show(10)

# Verificar que el resultado es el mismo
df_nativo.groupBy("categoria").count().orderBy("count", ascending=False).show()

## 8. Combinar DataFrames: Join y Union

### Join
Combina columnas de dos DataFrames basandose en una condicion (como SQL JOIN).

### Union
Apila las filas de dos DataFrames con el mismo schema (como SQL UNION ALL).

In [ ]:
# JOIN: combinar sales con stores para obtener informacion de la tienda
df_completo = df_sales.join(df_stores, on="Store", how="inner")

print("Schema despues del join:")
df_completo.printSchema()
df_completo.show(5)
print(f"Filas despues del join: {df_completo.count()}")

In [ ]:
# UNION: apilar DataFrames
# Ejemplo: dividir y volver a unir (para demostrar la sintaxis)
df_parte1 = df_sales.filter(F.col("Store") <= 22)
df_parte2 = df_sales.filter(F.col("Store") > 22)

print(f"Parte 1: {df_parte1.count()} filas")
print(f"Parte 2: {df_parte2.count()} filas")

df_unido = df_parte1.union(df_parte2)
print(f"Despues de union: {df_unido.count()} filas")
print(f"Original: {df_sales.count()} filas")

---
## Ejercicios

Ahora es tu turno de practicar. Completa los siguientes ejercicios.

In [ ]:
# =============================================================
# EJERCICIO 1: Limpiar flights.csv - eliminar filas con nulls
# =============================================================

# 1. Leer flights.csv
df_flights = spark.read.csv("/home/jovyan/datos/flights.csv", header=True, inferSchema=True)

# 2. Contar nulls en DEPARTURE_DELAY y ARRIVAL_DELAY
print("Valores nulos en columnas de retraso:")
df_flights.select(
    F.count(F.when(F.isnull("DEPARTURE_DELAY"), True)).alias("nulls_departure"),
    F.count(F.when(F.isnull("ARRIVAL_DELAY"), True)).alias("nulls_arrival")
).show()

# 3. Eliminar filas donde DEPARTURE_DELAY o ARRIVAL_DELAY sean null
filas_antes = df_flights.count()
df_limpio = df_flights.dropna(subset=["DEPARTURE_DELAY", "ARRIVAL_DELAY"])
filas_despues = df_limpio.count()

# 4. Mostrar cuantas filas se eliminaron
print(f"Filas antes de limpiar: {filas_antes:,}")
print(f"Filas despues de limpiar: {filas_despues:,}")
print(f"Filas eliminadas: {filas_antes - filas_despues:,}")
print(f"Porcentaje eliminado: {(filas_antes - filas_despues) / filas_antes * 100:.2f}%")

# Verificar que ya no hay nulls
print("\nVerificacion - nulls restantes:")
df_limpio.select(
    F.count(F.when(F.isnull("DEPARTURE_DELAY"), True)).alias("nulls_departure"),
    F.count(F.when(F.isnull("ARRIVAL_DELAY"), True)).alias("nulls_arrival")
).show()

> **Nota docente:** `dropna(subset=[...])` es la forma mas directa de eliminar filas
> con nulls en columnas especificas. Puntos importantes:
> - Si no se especifica `subset`, `dropna()` elimina filas con null en **cualquier** columna,
>   lo cual puede ser demasiado agresivo.
> - Un error comun es usar `df.filter(F.col("DEPARTURE_DELAY").isNotNull())` para una
>   sola columna, olvidando incluir la otra. Con `subset` se manejan ambas a la vez.
> - Siempre mostrar el conteo antes/despues para evaluar el impacto de la limpieza.
>   Si se elimina mas del 50% de los datos, quiza conviene usar `fillna()` en su lugar.

In [ ]:
# =============================================================
# EJERCICIO 2: Normalizar columnas de stores.csv
# =============================================================

# 1. Leer stores.csv
df_stores_ej = spark.read.csv("/home/jovyan/datos/stores.csv", header=True, inferSchema=True)
print("Schema original:")
df_stores_ej.printSchema()
df_stores_ej.show(5)

# 2. Renombrar columnas a snake_case
df_normalizado = df_stores_ej \
    .withColumnRenamed("Store", "store") \
    .withColumnRenamed("Type", "tipo") \
    .withColumnRenamed("Size", "tamano")

# 3. Convertir "tipo" a minusculas
df_normalizado = df_normalizado.withColumn("tipo", F.lower(F.col("tipo")))

# 4. Mostrar resultado
print("\nSchema normalizado:")
df_normalizado.printSchema()
print("Datos normalizados:")
df_normalizado.show()

> **Nota docente:** Normalizar nombres de columnas es una buena practica que facilita
> el trabajo posterior con SQL y evita errores por mayusculas/minusculas.
> - `withColumnRenamed` no modifica el DataFrame original (Spark es inmutable),
>   sino que retorna uno nuevo.
> - Se pueden encadenar multiples `withColumnRenamed` en una sola expresion.
> - `F.lower()` opera sobre el **contenido** de la columna, no sobre su nombre.
> - Un error comun es confundir renombrar la columna (cambiar el header) con
>   transformar su contenido (cambiar los valores). Aqui hacemos ambas cosas.

In [ ]:
# =============================================================
# EJERCICIO 3: UDF para categorizar distancia de vuelos
# =============================================================

# 1. Leer flights.csv
df_flights_udf = spark.read.csv("/home/jovyan/datos/flights.csv", header=True, inferSchema=True)

# 2. Crear la UDF para categorizar distancia
@udf(returnType=StringType())
def categorizar_distancia(distancia):
    if distancia is None:
        return "desconocido"
    elif distancia < 500:
        return "corto"
    elif distancia <= 1500:
        return "medio"
    else:
        return "largo"

# 3. Aplicar la UDF creando columna "tipo_vuelo"
df_con_tipo = df_flights_udf.withColumn("tipo_vuelo", categorizar_distancia(F.col("DISTANCE")))

# Mostrar ejemplos
print("Ejemplos de categorizacion:")
df_con_tipo.select("AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DISTANCE", "tipo_vuelo").show(10)

# 4. Distribucion por tipo de vuelo
print("Distribucion por tipo de vuelo:")
df_con_tipo.groupBy("tipo_vuelo").count().orderBy("count", ascending=False).show()

> **Nota docente:** Esta UDF se podria reemplazar perfectamente con `F.when().otherwise()`
> nativo, lo cual seria mas eficiente. El equivalente nativo seria:
> ```python
> df.withColumn("tipo_vuelo",
>     F.when(F.col("DISTANCE").isNull(), "desconocido")
>      .when(F.col("DISTANCE") < 500, "corto")
>      .when(F.col("DISTANCE") <= 1500, "medio")
>      .otherwise("largo")
> )
> ```
> El proposito del ejercicio es practicar la sintaxis de UDFs. En la practica real,
> reservar las UDFs para logica compleja que no se pueda expresar con funciones nativas
> (ej: llamadas a APIs externas, regex complejos, logica de negocio con muchas reglas).
> - Error comun: olvidar manejar `None` en la UDF, lo que causa `NullPointerException`.

---
## Desafio Extra (Opcional)

**Para estudiantes avanzados:**

Crea un pipeline completo de limpieza para `netflix_titles.csv`.

In [ ]:
# =============================================================
# DESAFIO: Pipeline completo de limpieza de netflix_titles.csv
# =============================================================

# 1. Leer netflix_titles.csv
df_netflix = spark.read.csv("/home/jovyan/datos/netflix_titles.csv", header=True, inferSchema=True)
print("=== ANTES DE LA LIMPIEZA ===")
print(f"Filas: {df_netflix.count()}")
print(f"Columnas: {len(df_netflix.columns)}")
df_netflix.printSchema()

# Contar nulls iniciales
print("\nNulls antes de limpiar:")
df_netflix.select(
    [F.count(F.when(F.isnull(c), c)).alias(c) for c in df_netflix.columns]
).show(truncate=False)

# 2. Manejar nulls
df_limpio = df_netflix.fillna({
    "director": "Desconocido",
    "cast": "No especificado",
    "country": "No especificado"
})

print("Nulls despues de fillna (director, cast, country):")
df_limpio.select(
    F.count(F.when(F.isnull("director"), True)).alias("director"),
    F.count(F.when(F.isnull("cast"), True)).alias("cast"),
    F.count(F.when(F.isnull("country"), True)).alias("country")
).show()

# 3. Parsear date_added a tipo fecha
df_limpio = df_limpio.withColumn(
    "date_added_parsed",
    F.to_date(F.trim("date_added"), "MMMM d, yyyy")
)

print("Ejemplo de parseo de fecha:")
df_limpio.select("date_added", "date_added_parsed").show(5, truncate=False)

# 4. Separar "duration" en valor numerico y unidad
df_limpio = df_limpio \
    .withColumn("duration_valor", F.split("duration", " ").getItem(0).cast(IntegerType())) \
    .withColumn("duration_unidad", F.split("duration", " ").getItem(1))

print("Columna duration separada:")
df_limpio.select("duration", "duration_valor", "duration_unidad").show(10)

# 5. Eliminar duplicados por show_id
filas_antes = df_limpio.count()
df_limpio = df_limpio.dropDuplicates(["show_id"])
filas_despues = df_limpio.count()
print(f"Duplicados por show_id eliminados: {filas_antes - filas_despues}")

# 6. Schema final y primeras 10 filas
print("\n=== DESPUES DE LA LIMPIEZA ===")
df_limpio.printSchema()
df_limpio.show(10, truncate=40)

# 7. Verificacion final de nulls
print("\nConteo de nulls final:")
df_limpio.select(
    [F.count(F.when(F.isnull(c), c)).alias(c) for c in df_limpio.columns]
).show(truncate=False)

> **Nota docente:** Este desafio integra todas las tecnicas vistas en la actividad.
> Puntos clave para la evaluacion:
> 
> 1. **fillna** es preferible a dropna aqui porque director/cast/country tienen muchos
>    nulls y perder esas filas significaria perder datos valiosos de otros campos.
> 2. **Parseo de fecha:** El `F.trim()` es importante porque `date_added` puede tener
>    espacios al inicio. Sin trim, `to_date` retornaria null para muchas filas.
>    El formato `"MMMM d, yyyy"` maneja meses escritos completos ("January", "February").
> 3. **split de duration:** `F.split("duration", " ")` separa "90 min" en ["90", "min"].
>    Luego `.getItem(0)` obtiene "90" y `.cast(IntegerType())` lo convierte a entero.
>    Un error comun es olvidar el `.cast()` y quedarse con el valor como string.
> 4. **dropDuplicates** por `show_id` asegura unicidad en la clave primaria.
> 5. El pipeline sigue un orden logico: nulls -> tipos -> duplicados -> verificacion.

---
## Resumen

En esta actividad aprendimos:

1. **Detectar nulls:** `F.count(F.when(F.isnull(c), c))` para contar nulls por columna
2. **Manejar nulls:** `dropna(subset=[...])` para eliminar, `fillna({...})` para rellenar
3. **Duplicados:** `dropDuplicates()` para eliminar registros repetidos
4. **Casting:** `F.to_date()`, `.cast()` para convertir tipos de datos
5. **UDFs:** `@udf(returnType=...)` para logica personalizada (usar con moderacion)
6. **when/otherwise:** Alternativa nativa y mas rapida que UDFs para logica condicional
7. **Join y Union:** Combinar DataFrames por columnas (join) o apilar filas (union)

In [ ]:
# Detener la SparkSession al finalizar
spark.stop()
print("SparkSession detenida correctamente.")